# Fine-Tune LLMs with QLoRA and Serve Them with vLLM

Step-by-step notebook: adapt a causal language model with QLoRA on a single GPU, then serve it with vLLM for high-throughput batched inference.

[Read the full article](https://jheiduk.com/posts/vllm-qlora-finetuning/)

> **Runtime:** Select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# Core ML libraries: transformers, quantization, LoRA adapters, training loop
!pip install -q transformers peft trl bitsandbytes accelerate datasets

# vLLM — high-throughput inference engine with PagedAttention
!pip install -q vllm

## Step 1 — Load the base model in 4-bit

We load TinyLlama-1.1B using BitsAndBytes NF4 quantization with double quantization enabled. This keeps the base model weights in ~0.7 GB of VRAM instead of ~2.2 GB in fp16.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NF4 is optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # TinyLlama has no dedicated pad token

print(f"Model loaded. Device map: {model.hf_device_map}")

## Step 2 — Attach LoRA adapters

We inject trainable low-rank matrices (rank=16) into the four attention projections. `prepare_model_for_kbit_training` enables gradient checkpointing and casts LayerNorm to fp32 for stable training.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                                               # rank — higher = more capacity
    lora_alpha=32,                                      # scaling: effective lr ∝ alpha/r
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # attention projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 3 — Train with SFTTrainer

We use 2,000 Python instruction-following examples from the Alpaca-formatted `iamtarun/python_code_instructions_18k_alpaca` dataset. SFTTrainer wraps the standard Trainer with supervised fine-tuning conveniences.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train[:2000]")

sft_config = SFTConfig(
    output_dir="./tinyllama-python-adapter",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    max_seq_length=512,
    dataset_text_field="output",
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()
trainer.model.save_pretrained("./tinyllama-python-adapter")
tokenizer.save_pretrained("./tinyllama-python-adapter")
print("Adapter saved to ./tinyllama-python-adapter")

## Step 4A — Merge adapter and serve with vLLM

For production use with a single adapter, merge the LoRA weights back into the base model. vLLM then serves the merged checkpoint with PagedAttention and continuous batching.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Load base in full precision and fuse LoRA weights (ΔW = BA)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
merged = PeftModel.from_pretrained(base, "./tinyllama-python-adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("./tinyllama-merged")
tokenizer.save_pretrained("./tinyllama-merged")
print("Merged model saved.")

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="./tinyllama-merged")
outputs = llm.generate(
    ["Write a Python function that reverses a linked list."],
    SamplingParams(temperature=0.7, max_tokens=256),
)
print(outputs[0].outputs[0].text)

## Step 4B — Dynamic LoRA loading in vLLM

When serving multiple adapters from a single base model, vLLM can load LoRA weights dynamically per request. This avoids keeping separate model copies in GPU memory.

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# One base model instance — adapters loaded on demand per request
llm_lora = LLM(model=MODEL_ID, enable_lora=True, max_lora_rank=64)

outputs = llm_lora.generate(
    ["Write a Python function that reverses a linked list."],
    SamplingParams(temperature=0.7, max_tokens=256),
    lora_request=LoRARequest(
        lora_name="python_adapter",
        lora_int_id=1,
        lora_local_path="./tinyllama-python-adapter",
    ),
)
print(outputs[0].outputs[0].text)